In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

data_path = Path('../../../data/digitization_intermediates/04_metadata_extraction/validation')
figures_path = Path('../../../manuscript/figures')
figures_path.mkdir(parents=True, exist_ok=True)

# Load category dataset for Panel C
cat_df = pd.read_excel(data_path / 'test_set_cat_grades.xlsx')
categories = cat_df['category'].tolist()
curated_category_acc = cat_df['curated_set'].tolist()
randomized_category_acc = cat_df['random_set'].tolist()

# Load individual dataset for Panels A & B
indiv_df = pd.read_excel(data_path / 'test_set_indiv_grades.xlsx')

# Remove rows with blank/NaN values and convert to percentages
curated_accuracy = indiv_df['curated_set'].dropna() * 100
randomized_accuracy = indiv_df['random_set'].dropna() * 100

print(f"Loaded {len(curated_accuracy)} curated data points")
print(f"Loaded {len(randomized_accuracy)} randomized data points")
print(f"Loaded {len(categories)} categories")

# Create the figure with subplots
fig = plt.figure(figsize=(16, 12))

# Top row: Panels A and B side by side
# Bottom row: Panel C spanning full width
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.3, wspace=0.2)

# Panel A: Histogram of curated dataset
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(curated_accuracy, bins=15, alpha=0.7, color='#1f77b4', edgecolor='black', linewidth=0.5)
ax1.set_title('Panel A: Curated Test Set Accuracy by Entry', fontsize=14, fontweight='bold')
ax1.set_xlabel('Accuracy (%)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_xlim(0, 100)
ax1.grid(True, alpha=0.3)
mean_curated = np.mean(curated_accuracy)
ax1.axvline(mean_curated, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_curated:.2g}%')
ax1.legend()

# Panel B: Histogram of randomized dataset
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(randomized_accuracy, bins=15, alpha=0.7, color='#ff7f0e', edgecolor='black', linewidth=0.5)
ax2.set_title('Panel B: Random Test Set Accuracy by Entry', fontsize=14, fontweight='bold')
ax2.set_xlabel('Accuracy (%)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_xlim(0, 100)
ax2.grid(True, alpha=0.3)
mean_randomized = np.mean(randomized_accuracy)
ax2.axvline(mean_randomized, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_randomized:.2g}%')
ax2.legend()

# Panel C: Lollipop chart by category
ax3 = fig.add_subplot(gs[1, :])

y_pos = np.arange(len(categories))
dot_offset = 0.15

ax3.axvline(x=80, color='green', linestyle='--', alpha=0.7, linewidth=2, label='80% threshold')
ax3.axvline(x=90, color='green', linestyle='--', alpha=0.7, linewidth=2, label='90% threshold')

for i, (curated_val, random_val) in enumerate(zip(curated_category_acc, randomized_category_acc)):
    ax3.plot([50, curated_val], [y_pos[i] - dot_offset, y_pos[i] - dot_offset],
             color='#1f77b4', linewidth=2, alpha=0.7)
    ax3.plot([50, random_val], [y_pos[i] + dot_offset, y_pos[i] + dot_offset],
             color='#ff7f0e', linewidth=2, alpha=0.7)
    ax3.scatter(curated_val, y_pos[i] - dot_offset, color='#1f77b4', s=100,
                zorder=3, label='Curated Test Set' if i == 0 else "")
    ax3.scatter(random_val, y_pos[i] + dot_offset, color='#ff7f0e', s=100,
                zorder=3, label='Random Test Set' if i == 0 else "")
    ax3.text(curated_val + 1, y_pos[i] - dot_offset, f'{curated_val:.2g}%',
             va='center', fontsize=9, color='black')
    ax3.text(random_val + 1, y_pos[i] + dot_offset, f'{random_val:.2g}%',
             va='center', fontsize=9, color='black')

ax3.set_xlabel('Accuracy (%)', fontsize=12)
ax3.set_title('Panel C: Accuracy by Category', fontsize=14, fontweight='bold')
ax3.set_yticks(y_pos)
ax3.set_yticklabels(categories)
ax3.legend(loc='lower right')
ax3.set_xlim(50, 100)
ax3.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(figures_path / 'metadata_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()